[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/06_advanced_ai/06_advanced_filtering.ipynb)


# 06 — 고급 필터링·자동화

> 본문: [`06_advanced_ai.md`](06_advanced_ai.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 `data/` 의 **실제 BoltzGen 실행 결과**에서 계산합니다(임의값 아님).
> **분석 셀 실행 3초.**

> 런타임 변경 없이 그대로 실행하세요.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 노트북은 **Colab과 로컬 모두**에서 동작합니다.

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`06_advanced_ai`) 폴더로 자동 이동한 뒤, `data/` 의 실제 결과로 실습합니다.
- **로컬**: 이 노트북을 `06_advanced_ai/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

> 런타임은 **기본값 그대로** 두면 됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "06_advanced_ai"
PIP_PKGS = "pandas matplotlib"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# 필요한 라이브러리 확보: import 안 되는 것만 설치(Colab 새 런타임/로컬 모두 자동 대응)
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# ── 내가 만든 결과 우선, 없으면 레퍼런스 ──────────────────────────────────────
#   MY  : 이 노트북에서 내가 직접 돌려 만든 산출물
#   REF : 저장소에 커밋된 레퍼런스 결과(대조군 · 실행을 건너뛰어도 실습이 이어지도록)
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """산출물 하나를 찾는다 — 내가 만든 my_run/ 트리를 먼저 뒤지고, 없으면 레퍼런스 폴더에서.

    파일명 규칙이 달라도(내 실행 rank1_*.cif / 레퍼런스 rank001_*.cif,
    final_<budget>_designs / final_designs) 같은 글롭으로 잡히도록 설계했습니다.
    """
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 이미 설계를 돌렸다면 그 결과를 이어받는다(이 챕터에서 다시 안 돌려도 됨).

    내 my_run/ 에 결과가 있으면 그대로 쓰고, 없으면 인자로 준 순서대로 앞 챕터를 찾는다.
    아무 데도 없으면 MY 를 그대로 둬서 find_one() 이 레퍼런스로 폴백하게 한다.
    """
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 내가 만든 그림은 my_ 접두어로 저장 — 저장소에 커밋된 레퍼런스 그림을 덮어쓰지 않도록.
def my_fig(name):
    return f"my_{name}"

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd())

## 1) 전체 디자인 메트릭 로드 (필터링 전)
`all_designs_metrics.csv` 는 선별 전 모든 디자인. 여기에 직접 필터를 실험합니다.
아래 표는 **일부 미리보기가 아니라 전체 행**이에요 — Colab 은 표 아래 페이지 컨트롤로 넘겨 볼 수 있고,
주피터는 긴 표를 앞뒤만 잘라 `N rows × M columns` 로 표시합니다.

In [ ]:
import pandas as pd
# Ch.05 또는 Ch.04 에서 직접 돌린 결과가 있으면 그걸 이어서 씁니다(먼저 찾는 쪽 우선)
inherit_run("../05_result_interpretation/my_run", "../04_basic_usage/my_run")
df = pd.read_csv(find_one("all_designs_metrics.csv", "data/vanilla"))
print("총 디자인:", len(df))
# .head() 를 쓰지 않습니다 — 위 제목대로 '전체'를 그대로 보여주려고요.
df[cols_in(df, "id", "design_ptm", "design_to_target_iptm", "filter_rmsd",
           "plip_hbonds_refolded")]

## 2) 맞춤 하드 필터 (실제 컬럼명)
ipTM>0.5 · pTM>0.7 · RMSD<2.0Å 동시 통과만 남깁니다.

In [ ]:
f = df[(df["design_to_target_iptm"] > 0.5) & (df["design_ptm"] > 0.7) & (df["filter_rmsd"] < 2.0)]
print("필터 통과:", len(f), "/", len(df))
f[["id", "design_to_target_iptm", "design_ptm", "filter_rmsd"]].sort_values(
    "design_to_target_iptm", ascending=False)

## 3) CLI 고급 필터 옵션 (본문 6.4) — pandas 실험을 그대로 명령으로
| 옵션 | 의미 |
|------|------|
| `--metrics_override k=w` | 메트릭 가중치(클수록 덜 중요) |
| `--additional_filters 'feat<2.0'` | 하드 필터 추가 |
| `--alpha 0.3` | 다양성↑(높을수록) vs 품질↑(낮을수록) |
| `--size_buckets 80-100:5` | 길이대별 할당 |

In [ ]:
cli = ("boltzgen run spec.yaml --output out --steps filtering \\\n"
       "  --metrics_override plip_hbonds_refolded=4 delta_sasa_refolded=2 \\\n"
       "  --additional_filters 'designfolding-filter_rmsd<2.0' \\\n"
       "  --alpha 0.3 --size_buckets 80-100:5 100-140:5")
print(cli)
print("\n핵심 패턴(본문 6.2): 무거운 design~analysis 는 한 번만, filtering 은 기준 바꿔가며 여러 번(초 단위).")

## 4) 파라미터 스윕 골격 + 결과 비교 (본문 6.3)
설정을 바꿔가며 돌리고, 평균 ipTM 으로 비교합니다. 여기선 **필터 전/후 그룹**으로 `compare_bars` 를 시연합니다.

In [ ]:
import itertools
for budget, num in itertools.product([20, 50], [1000, 5000]):
    print(f"boltzgen run design.yaml --budget {budget} --num_designs {num} --output sweep/b{budget}_n{num}")

In [ ]:
from boltzgen_viz import load_metrics, compare_bars
all_rows = load_metrics(str(find_one("all_designs_metrics.csv", "data/vanilla", quiet=True)))
passed   = [r for r in all_rows if r["design_to_target_iptm"] > 0.5
            and r["design_ptm"] > 0.7 and r["filter_rmsd"] < 2.0]
compare_bars({"all designs": all_rows, "hard-filtered": passed},
             "design_to_target_iptm", "Filtering Effect — mean ipTM",
             "mean ipTM", my_fig("06_filter_compare.png"), thr=0.5, thr_label="Good (0.5)")
from IPython.display import Image; Image(my_fig("06_filter_compare.png"))

## 5) 계층적 스크리닝 (본문 6.1)
Level1 (num_designs 1e4, budget 200) → 명세 제약 강화 → Level2 (5e3, budget 20) → 상위 5~10 실험.
**규모가 품질을 만든다** — 테스트는 작게, 프로덕션은 크게.